# Itô's Lemma, Intuitively

10/03 stated that the log of a GBM price drifts at $\mu - \sigma^2/2$, not $\mu$. Where does the $-\sigma^2/2$ come from? Ordinary calculus says $d\ln S = dS/S$, which would give a drift of $\mu$.
The answer is Itô's lemma: for functions of Brownian motion, the second-order Taylor term does not vanish, because $(dW)^2 = dt$ (10/02). This notebook
derives the lemma from a Taylor expansion, applies it to the standard examples, shows numerically why stochastic integrals must be defined with
left-point sums, and uses the lemma to explain the profit and loss of a delta-hedged option, the step that leads to Black–Scholes.

## 0. Math link

There is no stochastic calculus notebook in the course repos yet (curriculum gap). This notebook builds on:

- [Chain rule](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/00_derivatives/16_properties_of_the_derivative_chain_rule.ipynb)
  and [higher-order derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/09_higher_order_derivatives.ipynb):
  Itô's lemma is the chain rule with one extra second-order term.
- [Partial derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/01_partial_derivatives_part_1.ipynb) for functions of time and price.
- Quadratic variation of Brownian motion from [10/02](./02_from_random_walk_to_brownian_motion.ipynb); GBM from [10/03](./03_geometric_brownian_motion_and_lognormal_prices.ipynb).

## 1. Introduction

An option is a function of the underlying price and time, $V(t, S_t)$. To hedge it, price it, or explain its daily profit and loss, we need to know how $V$
changes when $S$ follows a random process. For smooth paths the chain rule would answer that. Brownian paths are too rough: over a short interval
$\Delta t$ the move $\Delta W$ is of size $\sqrt{\Delta t}$, so $(\Delta W)^2$ is of size $\Delta t$, the same order as the drift, and cannot be dropped.

After this notebook you can:

- derive Itô's lemma from a second-order Taylor expansion and the rule $(dW)^2 = dt$;
- apply it to $W^2$, $\ln S$, $S^2$ and functions of time and price;
- explain why the Itô integral uses left-point sums, and compute $\int_0^T W\,dW$;
- decompose the P&L of a delta-hedged option into gamma and theta, and relate it to realized and implied volatility.

## 2. Theory

### Why the ordinary chain rule fails

For a smooth function $f$ and a small change $\Delta x$: $f(x + \Delta x) - f(x) = f'(x)\Delta x + \tfrac12 f''(x)(\Delta x)^2 + \dots$. For a smooth path, $\Delta x \approx x'\Delta t$, so $(\Delta x)^2 \sim \Delta t^2$ and
the second term vanishes in the limit. For Brownian motion, $\Delta W \sim \sqrt{\Delta t}$ and $(\Delta W)^2$ has mean $\Delta t$. Summed over a path, the squared increments add up to the
elapsed time (quadratic variation, 10/02): their sum is not random in the limit. So in the limit

$$
(dW)^2 = dt, \qquad dt\,dW = 0, \qquad (dt)^2 = 0
$$

### Itô's lemma

Let $dX_t = a\,dt + b\,dW_t$ (drift $a$ and volatility $b$ may depend on $t$ and $X_t$), and let $f(t, x)$ be twice differentiable. Expanding to second order and using
$(dX)^2 = b^2dt$:

$$
df = \Big(\frac{\partial f}{\partial t} + a\frac{\partial f}{\partial x} + \frac12 b^2\frac{\partial^2 f}{\partial x^2}\Big)dt + b\frac{\partial f}{\partial x}\,dW
$$

The extra term $\frac12 b^2 f_{xx}$ is the **Itô correction**. It is a convexity effect: for convex $f$, randomness in $X$ pushes $f(X)$ up on average (Jensen's inequality).

### Standard examples

- $f = W^2$: $d(W^2) = dt + 2W\,dW$. The $dW$ term has mean zero, so $E[W_t^2] = t$, and $W_t^2 - t$ is a martingale.
- GBM, $dS = \mu S\,dt + \sigma S\,dW$, and $f = \ln S$ ($f_S = 1/S$, $f_{SS} = -1/S^2$):

$$
d\ln S = \Big(\mu - \frac12\sigma^2\Big)dt + \sigma\,dW
$$

  so $\ln S_t = \ln S_0 + (\mu - \sigma^2/2)t + \sigma W_t$: the solution used in 10/03.
- $f = S^2$ under GBM: $d(S^2) = (2\mu + \sigma^2)S^2dt + 2\sigma S^2dW$, so $E[S_t^2] = S_0^2e^{(2\mu + \sigma^2)t}$, consistent with the lognormal variance.

### The Itô integral

$\int_0^T f\,dW$ is defined as the limit of **left-point** sums $\sum_i f(W_{t_{i-1}})(W_{t_i} - W_{t_{i-1}})$: the integrand is fixed before the increment happens, like a trading position
chosen before the price move. With this choice:

$$
\int_0^T W\,dW = \frac12 W_T^2 - \frac12 T
$$

(integrate $d(W^2) = dt + 2W\,dW$). Midpoint sums converge to $\frac12W_T^2$ instead, the ordinary-calculus answer: for rough paths the evaluation point matters.
The left-point integral has mean zero, which is why gains from trading with a non-anticipating strategy in a martingale have mean zero.

### Delta hedging and the P&L of an option

Let $V(t, S)$ be an option value and hold the option, short $\Delta = V_S$ shares. By Itô's lemma under GBM,

$$
dV = V_t\,dt + V_S\,dS + \tfrac12\sigma^2S^2V_{SS}\,dt
$$

and the hedged position changes by

$$
d(V - \Delta S) = \big(V_t + \tfrac12\sigma^2S^2V_{SS}\big)dt
$$

The $dW$ term has disappeared: the hedge removes the randomness to first order. What remains is **theta** $V_t$ (time decay, usually negative for a long
option) and **gamma** $\Gamma = V_{SS}$ times $\frac12\sigma^2S^2$. Over a short step with realized move $\Delta S$, the gamma gain is $\frac12\Gamma(\Delta S)^2$. If the option was priced with volatility
$\sigma_{\text{imp}}$, so that theta offsets gamma exactly at that volatility, the daily P&L of the hedged option is approximately

$$
\tfrac12\Gamma S^2\big(r_{\text{day}}^2 - \sigma_{\text{imp}}^2\Delta t\big)
$$

A long, delta-hedged option gains when realized volatility exceeds implied volatility and loses otherwise. Setting the riskless hedged position to earn the
risk-free rate gives the Black–Scholes equation (10/05).

## 3. Worked Examples

### Example 1: $W^2$

$d(W^2) = 2W\,dW + dt$. Taking expectations: $dE[W^2] = dt$, so $E[W_t^2] = t$, as in 10/02.

### Example 2: the log price

$\mu = 7\%$, $\sigma = 20\%$: $d\ln S = (0.07 - 0.02)dt + 0.2\,dW$. The log price drifts at 5% a year, as found in 10/03.

### Example 3: $\int W\,dW$

On a path with $W_1 = 1.2$: Itô gives $\frac12(1.44 - 1) = 0.22$; ordinary calculus would give $0.72$.

### Example 4: $E[S^2]$

$S_0 = 100$, $\mu = 7\%$, $\sigma = 20\%$, $t = 1$: $E[S_1^2] = 10{,}000\,e^{0.18} = 11{,}972$, so $\operatorname{Var}(S_1) = 11{,}972 - (100e^{0.07})^2 = 11{,}972 - 11{,}503 = 469$, standard deviation 21.7, as the lognormal formula of 10/03 gives.

### Example 5: gamma and theta

An at-the-money option on a CHF 100 stock with gamma 0.02 per CHF and implied volatility 20%. Daily theta from the gamma side: $\frac12\cdot 0.02\cdot 0.04\cdot 100^2/252 = 0.0159$ CHF.
A 2% move (CHF 2) earns $\frac12\cdot 0.02\cdot 4 = 0.04$ CHF; break-even is a move of $100\cdot 0.2/\sqrt{252} = 1.26$ CHF (1.26%), one implied standard deviation.

## 4. Finance Application

### Where it matters

- **Option pricing.** The Black–Scholes equation follows from Itô's lemma applied to $V(t, S)$ and the hedging argument (10/05).
- **Trading desks.** Daily option P&L is explained as delta, gamma and theta terms; "long gamma, short theta" is the Itô decomposition.
- **Volatility trading.** Delta-hedged options and variance swaps are bets on realized versus implied volatility.
- **Interest-rate and credit models.** Bond prices, yields and survival probabilities are functions of stochastic factors; Itô's lemma gives their dynamics.

### The usual misreadings

- "If the stock's expected return is 7%, the expected log return is 7%." It is $7\% - \sigma^2/2$.
- "A hedged option has no risk." Delta hedging removes first-order risk; discrete rebalancing and the gap between realized and implied volatility remain.
- "The integral of $W\,dW$ is $W^2/2$." Only with ordinary calculus; the Itô integral subtracts $T/2$.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

### Left-point vs midpoint sums for $\int_0^1 W\,dW$

One fine Brownian path with $2^{16}$ steps, evaluated on coarser grids.

In [ ]:
n_fine = 2**16
dW = np.sqrt(1 / n_fine) * rng.standard_normal(n_fine)
W = np.concatenate([[0.0], np.cumsum(dW)])
WT = W[-1]
rows = []
for n in [2**k for k in range(2, 16, 2)]:
    Wn = W[:: n_fine // n]                         # path on n steps
    Wmid = W[n_fine // n // 2 :: n_fine // n]      # midpoints of the coarse intervals
    inc = np.diff(Wn)
    rows.append({"steps": n, "left-point sum": np.sum(Wn[:-1] * inc), "midpoint sum": np.sum(Wmid[:n] * inc),
                 "right-point sum": np.sum(Wn[1:] * inc)})
tbl = pd.DataFrame(rows).set_index("steps")
print(f"W_1 = {WT:.4f};  Ito (W^2 - 1)/2 = {(WT**2 - 1) / 2:.4f};  ordinary W^2/2 = {WT**2 / 2:.4f}")
tbl.round(4)

On this path $W_1 = -0.378$. As the grid gets finer, the left-point sums settle near the Itô value $(W_1^2 - 1)/2 = -0.429$, the midpoint sums near the ordinary-calculus
value $W_1^2/2 = 0.071$, and the right-point sums near $(W_1^2 + 1)/2 = 0.571$. The three differ by multiples of $T/2$: the sum of squared increments.

### Checking the GBM solution

Simulate $dS = \mu S\,dt + \sigma S\,dW$ step by step (Euler, 10,000 steps per year) and compare $\ln S_1$ with the Itô solution $\ln S_0 + (\mu - \sigma^2/2) + \sigma W_1$ and with the naive
$\ln S_0 + \mu + \sigma W_1$.

In [ ]:
S0, mu, sigma = 100.0, 0.07, 0.20
n_paths, n_steps = 5_000, 10_000
dt = 1 / n_steps
S = np.full(n_paths, S0)
Wsum = np.zeros(n_paths)
for _ in range(n_steps):
    dW_ = np.sqrt(dt) * rng.standard_normal(n_paths)
    S = S * (1 + mu * dt + sigma * dW_)
    Wsum += dW_
ito = np.log(S0) + (mu - 0.5 * sigma**2) + sigma * Wsum
naive = np.log(S0) + mu + sigma * Wsum
print(f"mean |ln S_1 - Ito|   = {np.mean(np.abs(np.log(S) - ito)):.5f}")
print(f"mean |ln S_1 - naive| = {np.mean(np.abs(np.log(S) - naive)):.5f}  (sigma^2/2 = {0.5 * sigma**2:.5f})")
print(f"E[S_1^2]: simulated {np.mean(S**2):,.0f}, Ito {S0**2 * np.exp(2 * mu + sigma**2):,.0f}")

The step-by-step simulation agrees with the Itô solution to within 0.0002 in log terms; the naive solution with drift $\mu$ is off by exactly $\sigma^2/2 = 0.02$ on every path.

### Delta-hedged option: gamma against theta

Buy a one-year at-the-money call priced at implied volatility 20%, hedge its delta daily with the stock (Black–Scholes delta, 10/05), and finance at the
risk-free rate. Simulate paths whose **realized** volatility is 15%, 20% or 25%.

In [ ]:
def bs_call(S, K, tau, r, sig):
    tau = np.maximum(tau, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))
    d2 = d1 - sig * np.sqrt(tau)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * tau) * stats.norm.cdf(d2), stats.norm.cdf(d1), \
        stats.norm.pdf(d1) / (S * sig * np.sqrt(tau))


def hedged_pnl(sig_real, sig_imp=0.20, r=0.01, K=100.0, n_days=252, n_paths=2_000, rng_=rng):
    dt_ = 1 / n_days
    S_ = np.full(n_paths, 100.0)
    V, delta, gamma = bs_call(S_, K, 1.0, r, sig_imp)
    cash = -V + delta * S_                          # bought the option, sold delta shares
    gamma_terms = np.zeros(n_paths)
    for i in range(1, n_days + 1):
        z = rng_.standard_normal(n_paths)
        S_new = S_ * np.exp((r - 0.5 * sig_real**2) * dt_ + sig_real * np.sqrt(dt_) * z)
        ret = S_new / S_ - 1
        gamma_terms += 0.5 * gamma * S_**2 * (ret**2 - sig_imp**2 * dt_) * np.exp(r * (1 - i * dt_))
        cash = cash * np.exp(r * dt_)
        S_ = S_new
        tau = 1.0 - i * dt_
        if i < n_days:
            _, new_delta, gamma = bs_call(S_, K, tau, r, sig_imp)
            cash += (new_delta - delta) * S_            # sell more shares when delta rises
            delta = new_delta
    payoff = np.maximum(S_ - K, 0)
    pnl = payoff - delta * S_ + cash
    return pnl, gamma_terms

res = {}
for sr in [0.15, 0.20, 0.25]:
    pnl, gt = hedged_pnl(sr)
    res[sr] = (pnl, gt)
    print(f"realized vol {sr:.0%}: mean P&L {pnl.mean():6.3f} CHF, std {pnl.std():.3f};  "
          f"sum of gamma terms {gt.mean():6.3f};  corr(P&L, gamma terms) {np.corrcoef(pnl, gt)[0, 1]:.3f}")
price0 = bs_call(100.0, 100.0, 1.0, 0.01, 0.20)[0]
print(f"option price at 20% implied vol: {price0:.3f} CHF")

The call costs CHF 8.43 at 20% implied volatility. Hedged daily, it earns on average −1.98 CHF if realized volatility is 15%, nothing if it is 20%, and +1.97 CHF if it is
25%. The P&L of each path is almost exactly the sum of the gamma terms $\frac12\Gamma S^2(r^2 - \sigma_{\text{imp}}^2\Delta t)$ (correlation above 0.99): the stock's direction does not matter, only the size
of its moves relative to the implied volatility. Even when realized equals implied, discrete daily hedging leaves a P&L standard deviation of CHF 0.43.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(tbl.index, tbl["left-point sum"], "o-", label="left-point sums (Itô)")
ax.semilogx(tbl.index, tbl["midpoint sum"], "s-", label="midpoint sums")
ax.semilogx(tbl.index, tbl["right-point sum"], "^-", label="right-point sums")
ax.axhline((WT**2 - 1) / 2, color="tab:blue", linestyle="--", label="(W² - T)/2")
ax.axhline(WT**2 / 2, color="tab:orange", linestyle="--", label="W²/2")
ax.axhline((WT**2 + 1) / 2, color="tab:green", linestyle="--", label="(W² + T)/2")
ax.set_xticks(tbl.index)
ax.set_xticklabels(tbl.index)
ax.set_title("Riemann sums of W dW on one path: the evaluation point matters")
ax.set_xlabel("number of steps")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(ito[:2_000], np.log(S[:2_000]), s=3, alpha=0.5, label="vs Itô solution")
ax.scatter(naive[:2_000], np.log(S[:2_000]), s=3, alpha=0.5, label="vs naive mu instead of mu - sigma²/2")
lims = [np.log(S).min(), np.log(S).max()]
ax.plot(lims, lims, color="black", linewidth=0.8)
ax.set_title("ln S_1 from step-by-step simulation vs closed forms")
ax.set_xlabel("closed form")
ax.set_ylabel("simulated ln S_1")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for sr, c in [(0.15, "tab:blue"), (0.20, "grey"), (0.25, "tab:red")]:
    axes[0].hist(res[sr][0], bins=50, histtype="step", linewidth=1.5, color=c, label=f"realized {sr:.0%}")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("P&L of a delta-hedged call bought at 20% implied vol")
axes[0].set_xlabel("P&L at expiry (CHF)")
axes[0].legend()
pnl25, gt25 = res[0.25]
axes[1].scatter(gt25, pnl25, s=4, alpha=0.4)
lim = [min(gt25.min(), pnl25.min()), max(gt25.max(), pnl25.max())]
axes[1].plot(lim, lim, color="black", linewidth=0.8)
axes[1].set_title("Realized 25%: P&L vs sum of ½ Γ S² (r² - σ²_imp Δt)")
axes[1].set_xlabel("sum of gamma terms (CHF)")
axes[1].set_ylabel("hedged P&L (CHF)")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Apply Itô's lemma to $f(W) = W^3$. What is $E[W_t^3]$, and which combination of $W_t^3$ and $\int W\,dt$ is a martingale?
2. Apply Itô's lemma to $f(S) = S^2$ for GBM and derive $E[S_t^2]$. Check it against the lognormal formula of 10/03 for $S_0 = 50$, $\mu = 5\%$, $\sigma = 30\%$, $t = 2$.
3. A trader is long a delta-hedged straddle and "loses theta every day". **Interpret** when this position makes money, in terms of realized and implied volatility,
   and what happens to its P&L in a calm market.

### Intermediate

4. Show with Itô's lemma that $M_t = \exp(\sigma W_t - \frac12\sigma^2t)$ satisfies $dM = \sigma M\,dW$ and is a martingale. Check $E[M_t] = 1$ by simulation for $\sigma = 0.5$ and $t = 1, 4$.
5. For the delta-hedged call, rebalance weekly instead of daily (realized = implied = 20%). How does the standard deviation of the P&L change? Relate it to the
   number of rebalancing dates.

### Advanced

6. Ornstein–Uhlenbeck process (a model for interest rates or spreads): $dX = -\kappa X\,dt + \sigma\,dW$. Apply Itô's lemma to $e^{\kappa t}X_t$ to solve it, and show $\operatorname{Var}(X_t) = \frac{\sigma^2}{2\kappa}(1 - e^{-2\kappa t})$ for
   $X_0 = 0$. Check by simulation with $\kappa = 2$, $\sigma = 0.3$.
7. Product rule: for $dX = a\,dt + b\,dW_1$ and $dY = c\,dt + e\,dW_2$ with $dW_1dW_2 = \rho\,dt$, $d(XY) = X\,dY + Y\,dX + \rho be\,dt$. Use it to compute $E[S^{(1)}_tS^{(2)}_t]$ for two GBMs with correlation $\rho$ and check by simulation.

## 8. Solutions / Checks

In [ ]:
# 1.
Wt = np.sqrt(2.0) * rng.standard_normal(200_000)
print(f"1. E[W_2^3] simulated {np.mean(Wt**3):.4f} (theory 0)")

# 2.
S0_, mu_, s_, t_ = 50.0, 0.05, 0.30, 2.0
e_ito = S0_**2 * np.exp((2 * mu_ + s_**2) * t_)
var_logn = S0_**2 * np.exp(2 * mu_ * t_) * (np.exp(s_**2 * t_) - 1)
print(f"2. E[S^2] Ito {e_ito:.2f};  from lognormal: Var + mean^2 = {var_logn + (S0_ * np.exp(mu_ * t_))**2:.2f}")

# 4.
for t4 in [1.0, 4.0]:
    w4 = np.sqrt(t4) * rng.standard_normal(1_000_000)
    print(f"4. t = {t4}: E[M_t] simulated {np.mean(np.exp(0.5 * w4 - 0.125 * t4)):.4f}")

In [ ]:
# 5.
def hedged_pnl_freq(rebal_every, n_days=252, n_paths=2_000, sig=0.20, r=0.01, K=100.0, rng_=rng):
    dt_ = 1 / n_days
    S_ = np.full(n_paths, 100.0)
    V, delta, _ = bs_call(S_, K, 1.0, r, sig)
    cash = -V + delta * S_
    for i in range(1, n_days + 1):
        S_ = S_ * np.exp((r - 0.5 * sig**2) * dt_ + sig * np.sqrt(dt_) * rng_.standard_normal(n_paths))
        cash = cash * np.exp(r * dt_)
        if i < n_days and i % rebal_every == 0:
            _, nd, _ = bs_call(S_, K, 1.0 - i * dt_, r, sig)
            cash += (nd - delta) * S_
            delta = nd
    return np.maximum(S_ - K, 0) - delta * S_ + cash

sd_d, sd_w = hedged_pnl_freq(1).std(), hedged_pnl_freq(5).std()
print(f"5. P&L std: daily {sd_d:.3f}, weekly {sd_w:.3f}, ratio {sd_w / sd_d:.2f} (sqrt(5) = {np.sqrt(5):.2f})")

# 6.
kap, s6, T6, n6 = 2.0, 0.3, 1.0, 1_000
X = np.zeros(20_000)
d6 = T6 / n6
for _ in range(n6):
    X = X - kap * X * d6 + s6 * np.sqrt(d6) * rng.standard_normal(X.size)
print(f"6. Var(X_1) simulated {X.var():.5f}, theory {s6**2 / (2 * kap) * (1 - np.exp(-2 * kap * T6)):.5f}")

# 7.
rho7, m1, m2, v1, v2 = 0.5, 0.06, 0.04, 0.2, 0.3
z1 = rng.standard_normal(500_000)
z2 = rho7 * z1 + np.sqrt(1 - rho7**2) * rng.standard_normal(500_000)
S1 = np.exp((m1 - 0.5 * v1**2) + v1 * z1)
S2 = np.exp((m2 - 0.5 * v2**2) + v2 * z2)
print(f"7. E[S1 S2] simulated {np.mean(S1 * S2):.4f}, Ito {np.exp(m1 + m2 + rho7 * v1 * v2):.4f}")

**1.** $d(W^3) = 3W^2dW + 3W\,dt$. Expectations: $dE[W^3] = 3E[W]\,dt = 0$, so $E[W_t^3] = 0$; and $W_t^3 - 3\int_0^t W_s\,ds$ is a martingale.

**2.** $E[S_t^2] = S_0^2e^{(2\mu + \sigma^2)t}$; with the numbers given, $2{,}500\,e^{0.38} = 3{,}656$, equal to the lognormal variance plus the squared mean.

**3.** The position earns $\frac12\Gamma(\Delta S)^2$ on every move and pays theta, which is set by the implied volatility. It makes money if the realized volatility over its life exceeds the
implied volatility it was bought at, and loses in a calm market, where the daily moves are too small to pay for the time decay. It is a bet on realized vs implied
volatility, not on the direction of the stock.

**4.** With $f(t, w) = e^{\sigma w - \sigma^2t/2}$: $f_t = -\frac12\sigma^2f$, $f_w = \sigma f$, $f_{ww} = \sigma^2f$, so $dM = (-\frac12\sigma^2M + \frac12\sigma^2M)dt + \sigma M\,dW = \sigma M\,dW$: no drift, a martingale with $E[M_t] = M_0 = 1$.

**5.** Weekly rebalancing raises the P&L standard deviation from 0.45 to 0.95 CHF, a factor of 2.1, close to $\sqrt5 = 2.24$: the hedging error is a sum of independent gamma terms, one per rebalancing interval, each with standard
deviation proportional to the interval length, so the total standard deviation scales with the square root of the interval.

**6.** $d(e^{\kappa t}X) = e^{\kappa t}(\kappa X\,dt + dX) = \sigma e^{\kappa t}dW$, so $X_t = \sigma\int_0^te^{-\kappa(t-s)}dW_s$ and $\operatorname{Var}(X_t) = \sigma^2\int_0^te^{-2\kappa(t-s)}ds = \frac{\sigma^2}{2\kappa}(1 - e^{-2\kappa t})$. The simulation agrees.

**7.** $d(S^{(1)}S^{(2)}) = S^{(1)}S^{(2)}\big[(\mu_1 + \mu_2 + \rho\sigma_1\sigma_2)dt + \sigma_1dW_1 + \sigma_2dW_2\big]$, so $E[S^{(1)}_tS^{(2)}_t] = S^{(1)}_0S^{(2)}_0e^{(\mu_1 + \mu_2 + \rho\sigma_1\sigma_2)t}$; the simulation matches.

## 9. Conclusion

- Over short intervals $(\Delta W)^2 \approx \Delta t$; in the limit $(dW)^2 = dt$, so the second-order Taylor term survives.
- Itô's lemma: $df = (f_t + af_x + \frac12b^2f_{xx})dt + bf_x\,dW$; the extra term is a convexity correction.
- It gives $d\ln S = (\mu - \sigma^2/2)dt + \sigma\,dW$ for GBM, $E[W_t^2] = t$, and $\int_0^TW\,dW = (W_T^2 - T)/2$.
- Itô integrals use left-point sums, like trading positions set before the price moves; midpoint sums give a different answer.
- A delta-hedged option's P&L is gamma minus theta, $\frac12\Gamma S^2(r^2 - \sigma_{\text{imp}}^2\Delta t)$ per step: it profits when realized volatility exceeds implied volatility.

Next: [10/05 Black–Scholes and Monte Carlo pricing](./05_black_scholes_and_monte_carlo_pricing.ipynb).